In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
import os
os.chdir("/content/drive/My Drive/YOLOV11BreastCancer/")

In [3]:
!pip install ultralytics
!pip install thop

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.4 MB/s eta 0:00:00


In [4]:
# %%
from yolo_config import Config
from yolo_file_manager import FileManager
from yolo_model import YoloModel
from yolo_metrics import MetricsAggregator
import os
import time
import torch
import pandas as pd
from thop import profile
from ultralytics import YOLO
# %%

os.chdir("/content/drive/My Drive/")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
if __name__ == '__main__':
    Config.set_gpu_settings()

    # -----------------------
    # Model Training
    # -----------------------
    for model_size in Config.MODEL_SIZES:
        start_time = time.time()

        # Initialize components
        file_manager = FileManager(model_size)
        yolo_model = YoloModel(file_manager)
        metrics_aggregator = MetricsAggregator(file_manager)

        # Configure resume training parameters
        resume_fold = 6  # El fold donde se detuvo el entrenamiento
        resume_epoch = 0  # La época desde donde reanudar
        is_resuming = True  # Cambiar a False para iniciar entrenamiento desde cero

        # NO limpiar runs previos cuando estamos reanudando
        if not is_resuming:
            file_manager.clean_model_runs()
            performance_metrics = []
        else:
            # Intentar cargar métricas de rendimiento existentes
            metrics_dir = os.path.join(Config.RUNS_DIR, Config.DATASET, model_size)
            performance_path = os.path.join(metrics_dir, f'performance_metrics_{Config.DATASET}.csv')

            if os.path.exists(performance_path):
                existing_metrics = pd.read_csv(performance_path)
                performance_metrics = existing_metrics.to_dict('records')
                print(f"Loaded {len(performance_metrics)} existing performance records")
            else:
                performance_metrics = []
                print("No existing performance metrics found, starting new records")

            # Cargar métricas de los folds anteriores
            for threshold in Config.THRESHOLDS:
                threshold_metrics_path = os.path.join(metrics_dir, f'metrics_{threshold}.csv')
                if os.path.exists(threshold_metrics_path):
                    try:
                        fold_metrics = pd.read_csv(threshold_metrics_path)
                        # Cargar estas métricas en metrics_aggregator
                        for index, row in fold_metrics.iterrows():
                            fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
                            if fold_name.startswith('fold_') and int(fold_name.split('_')[1]) < resume_fold:
                                metrics_data = {col: row[col] for col in fold_metrics.columns[1:]}
                                metrics_aggregator.metrics_by_threshold[threshold].loc[fold_name] = pd.Series(metrics_data)
                        print(f"Loaded threshold {threshold} metrics for previous folds")
                    except Exception as e:
                        print(f"Error loading metrics for threshold {threshold}: {e}")

        # Measure FLOPs and parameters (solo si no estamos reanudando o no tenemos estas métricas)
        if not is_resuming or not performance_metrics or 'model_specs' not in [m.get('phase') for m in performance_metrics]:
            weights = file_manager.get_pretrained_weights()
            model = YOLO(weights)
            torch_model = model.model
            torch_model = torch_model.to(Config.DEVICE)
            input_tensor = torch.randn(1, 3, Config.IMG_SIZE, Config.IMG_SIZE).to(Config.DEVICE)
            flops, params = profile(torch_model, inputs=(input_tensor,))
            print(f"Model: {model_size}")
            print(f"FLOPs: {flops/1e9:.2f}G, Parameters: {params/1e6:.2f}M")

            # Add model specs to metrics
            performance_metrics.append({
                'model': model_size,
                'phase': 'model_specs',
                'fold': 'N/A',
                'time_seconds': None,
                'memory_mb': None,
                'flops_g': flops/1e9,
                'params_m': params/1e6
            })

        # Training phase
        start_train_time = time.time()

        # Preparar ruta a los pesos de reanudación
        if is_resuming:
            # Construir la ruta según tu estructura
            resume_weights_path = os.path.join(Config.RUNS_DIR, Config.DATASET, model_size,
                                          f"fold_{resume_fold}/train/weights/epoch{resume_epoch}.pt")
            print(f"Attempting to resume from: {resume_weights_path}")
            if not os.path.exists(resume_weights_path):
                print(f"WARNING: Weight file not found: {resume_weights_path}")
                # Intentar buscar el último checkpoint disponible
                weights_dir = os.path.join(Config.RUNS_DIR, Config.DATASET, model_size, f"fold_{resume_fold}/train/weights")
                if os.path.exists(weights_dir):
                    checkpoint_files = [f for f in os.listdir(weights_dir) if f.startswith("epoch") and f.endswith(".pt")]
                    if checkpoint_files:
                        # Ordenar por número de época y tomar el último
                        checkpoint_files.sort(key=lambda x: int(x.replace("epoch", "").replace(".pt", "")))
                        resume_weights_path = os.path.join(weights_dir, checkpoint_files[-1])
                        print(f"Using latest available checkpoint: {resume_weights_path}")
                    else:
                        print("No checkpoint files found. Will start fold from scratch.")
                        resume_weights_path = None
                else:
                    print(f"Weights directory not found: {weights_dir}")
                    resume_weights_path = None

        # -----------------------
        # K-fold Cross Validation
        # -----------------------
        for fold in Config.FOLDS:
            # Skip completed folds if resuming
            if is_resuming and fold < resume_fold:
                print(f"Skipping completed fold: {fold}")
                continue

            # Reset memory stats for this fold
            torch.cuda.reset_peak_memory_stats()

            # Dataset setup
            file_manager.set_validation_setup(fold)

            # -----------------------
            # Training
            # -----------------------
            start_fold_time = time.time()

            if is_resuming and fold == resume_fold and resume_weights_path and os.path.exists(resume_weights_path):
                print(f"Resuming training for fold {fold} from {resume_weights_path}")
                yolo_model.train(resume=True, resume_weights=resume_weights_path)
            else:
                # Train normally for this fold
                yolo_model.train()

            end_fold_time = time.time()

            # Measure memory usage and time
            fold_time = end_fold_time - start_fold_time
            fold_memory = torch.cuda.max_memory_allocated() / 1e6  # MB
            print(f"Training time fold - {fold}: {fold_time:.2f} seconds")
            print(f"Memory used in fold - {fold}: {fold_memory:.2f} MB")

            # Record metrics
            performance_metrics.append({
                'model': model_size,
                'phase': 'training',
                'fold': fold,
                'time_seconds': fold_time,
                'memory_mb': fold_memory,
                'flops_g': None,
                'params_m': None
            })

            # -----------------------
            # Validation
            # -----------------------
            start_validation_time = time.time()

            # Evaluar el modelo con cada threshold
            for threshold in Config.THRESHOLDS:
                valid_metrics = yolo_model.validate(threshold)
                metrics_aggregator.add_metrics(threshold, valid_metrics, model_size)
                print(f"Fold {fold}, Threshold {threshold} - Metrics: {valid_metrics}")

            end_validation_time = time.time()
            validation_time = end_validation_time - start_validation_time

            # Record validation metrics
            performance_metrics.append({
                'model': model_size,
                'phase': 'validation',
                'fold': fold,
                'time_seconds': validation_time,
                'memory_mb': None,
                'flops_g': None,
                'params_m': None
            })

            # Save metrics after each fold to prevent data loss
            fold_metrics_dir = os.path.join(Config.RUNS_DIR, Config.DATASET, model_size)
            os.makedirs(fold_metrics_dir, exist_ok=True)

            # Guardar métricas de rendimiento incrementalmente
            fold_performance_path = os.path.join(fold_metrics_dir, f'performance_metrics_{Config.DATASET}.csv')
            pd.DataFrame(performance_metrics).to_csv(fold_performance_path, index=False)
            print(f"Intermediate performance metrics saved to {fold_performance_path}")

            # Clean up the weights after each fold
            file_manager.clean_weights()

        # Get the total training time
        total_train_time = time.time() - start_train_time
        print(f"Total training time for all folds: {total_train_time:.2f} seconds")

        # Aggregate Metrics
        metrics_aggregator.finish_validation()
        metrics_aggregator.save_metrics()

        # Reset memory stats for testing
        torch.cuda.reset_peak_memory_stats()

        # -----------------------
        # Testing
        # -----------------------
        start_test_time = time.time()
        file_manager.set_testing_setup()
        yolo_model.train()
        end_test_time = time.time()

        # Get the total testing time and memory
        test_time = end_test_time - start_test_time
        test_memory = torch.cuda.max_memory_allocated() / 1e6  # MB
        print(f"Test time: {test_time:.2f} seconds")
        print(f"Memory used in test: {test_memory:.2f} MB")

        # Record test metrics
        performance_metrics.append({
            'model': model_size,
            'phase': 'testing',
            'fold': 'N/A',
            'time_seconds': test_time,
            'memory_mb': test_memory,
            'flops_g': None,
            'params_m': None
        })

        # Save final performance metrics to CSV
        metrics_dir = os.path.join(Config.RUNS_DIR, Config.DATASET, model_size)
        performance_path = os.path.join(metrics_dir, f'performance_metrics_{Config.DATASET}_final.csv')
        pd.DataFrame(performance_metrics).to_csv(performance_path, index=False)
        print(f"Final performance metrics saved to {performance_path}")

Loaded 11 existing performance records


/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el no

Loaded threshold 0.001 metrics for previous folds


/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el no

Loaded threshold 0.1 metrics for previous folds


...

Loaded threshold 0.7 metrics for previous folds


/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el no

Loaded threshold 0.8 metrics for previous folds


/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el nombre del fold
/tmp/ipython-input-201028630.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fold_name = row[0]  # Asumiendo que el índice es el no

Streaming output truncated to the last 5000 lines.
     74/200      18.1G     0.0112      2.142      1.596         15        608: 100% ━━━━━━━━━━━━ 64/64 2.3it/s 28.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.0it/s 1.0s
                   all        100        107      0.661      0.542      0.565      0.259

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     75/200        18G    0.01114      2.039      1.528         12        448: 100% ━━━━━━━━━━━━ 64/64 2.5it/s 25.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.0it/s 1.0s
                   all        100        107      0.776      0.523       0.57       0.27

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     76/200      20.2G    0.01122      2.126      1.607         11        960: 100% ━━━━━━━━━━━━ 64/64 2.2it/s 29.5s
            